In [ ]:
from pyspark.sql import SparkSession

# where the csv is
CSV_PATH = "sample_data/steam-200k.csv"
# how many partitions for hash strategy
NUM_PARTITIONS_HASH = 8
# repartition to this many then coalesce down
REPARTITION_N = 16
COALESCE_N = 4


def print_table(title, rows):
    # rows = list of (game_title, hours), prints a bordered table
    if not rows:
        print(title)
        print("(no rows)")
        return
    col1, col2 = "game_title", "total_play_hours"
    w1 = max(len(col1), min(50, max(len(str(g)) for g, _ in rows)))
    w2 = max(len(col2), max(len(f"{h:.1f}") for _, h in rows))
    sep = "+" + "-" * (w1 + 2) + "+" + "-" * (w2 + 2) + "+"
    print(title)
    print(sep)
    print("| " + col1.ljust(w1) + " | " + col2.ljust(w2) + " |")
    print(sep)
    for game, hours in rows:
        g = (game[:47] + "...") if len(game) > 50 else game
        print("| " + g.ljust(w1) + " | " + f"{hours:.1f}".rjust(w2) + " |")
    print(sep)


# start spark
spark = SparkSession.builder.appName("ourdata").getOrCreate()
sc = spark.sparkContext

# load csv, no header, handle quoted game names
df = (
    spark.read.option("header", False)
    .option("quote", '"')
    .csv(CSV_PATH)
)

# give columns proper names and make value a number
df = df.select(
    df["_c0"].alias("user_id"),
    df["_c1"].alias("game_title"),
    df["_c2"].alias("behavior_name"),
    df["_c3"].cast("double").alias("value"),
)
# turn it into an rdd so we can use rdd stuff
rdd = df.rdd

# only keep "play" rows, then make (game, hours) pairs
play_rdd = rdd.filter(lambda row: row["behavior_name"] == "play")
pair_rdd = play_rdd.map(lambda row: (row["game_title"], row["value"]))

# strategy 1: hash partition by game title
hash_partitioned = pair_rdd.partitionBy(NUM_PARTITIONS_HASH)
# add up hours per game, sort by total, take top 20
result1 = (
    hash_partitioned.reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .take(20)
)

print_table("Strategy 1 (hash by game_title): top 20 games by total play hours", result1)

# strategy 2: repartition then coalesce
repartitioned = play_rdd.repartition(REPARTITION_N).coalesce(COALESCE_N)
pair_rdd2 = repartitioned.map(lambda row: (row["game_title"], row["value"]))
result2 = (
    pair_rdd2.reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .take(20)
)

print_table("\nStrategy 2 (repartition + coalesce): top 20 games by total play hours", result2)

# shut down spark
spark.stop()


Strategy 1 (hash by game_title): top 20 games by total play hours
+---------------------------------------------+------------------+
| game_title                                  | total_play_hours |
+---------------------------------------------+------------------+
| Dota 2                                      |         981684.6 |
| Counter-Strike Global Offensive             |         322771.6 |
| Team Fortress 2                             |         173673.3 |
| Counter-Strike                              |         134261.1 |
| Sid Meier's Civilization V                  |          99821.3 |
| Counter-Strike Source                       |          96075.5 |
| The Elder Scrolls V Skyrim                  |          70889.3 |
| Garry's Mod                                 |          49725.3 |
| Call of Duty Modern Warfare 2 - Multiplayer |          42009.9 |
| Left 4 Dead 2                               |          33596.7 |
| Football Manager 2013                       |          32308.